# 21 homa loopback

Question: does the engine of the crate `homa` deliver every message whole, in SRPT order, within
its grants and its limits, over a perfect link, on every point to point profile with the compact
form and on the Network profile with the full form.

The side under test is `aosabi-devtool homa-sim`, which runs two endpoints of the crate over the
loopback of `homa::sim`, end A the client and end B the server, and reports what each end reported
and a trace of every packet decision. The reference side is this notebook, which knows every byte it
asked each RPC to carry and checks the SHA-256 of what each end read against it, and
`tests/harness/homa/trace.py`, which asserts the properties of the engine over the trace, SRPT order
outside the FIFO share, every GRANT to the one message the scheduler names, the bytes granted and
not received against `window` and the bytes a receiver expects against `max_incoming`, at every row
under the `max_incoming` of the design note, the sum of the slot capacities the link may fill, and
at every GRANT under a tighter bound that binds the grants.

The notebook runs one half per entry of the matrix, which `half` names. The compact half is stage
H0's, over every point to point profile. The Network half is stage H1's, with the full form, the
Network profile of the design note, second version, section 3.2, in both scheduling modes, the
ports of a peer and the table at 4,096 peers. The real socket half is stage H2's, with the Unix
Socket transports of `homa::unix` in both forms, a `SOCK_SEQPACKET` socket and a stream socket in
prefixed AoFrame, the engine of the crate against the Python engine `tests/harness/homa/client.py`
in both directions, and the table against AoFrame alone on both forms. A half an entry does not
name prints not run below, which is never a pass, since it runs as the matrix entry of its own.

The exit number of the compact half, stage H0 of the plan's section The notebooks: 10,000 seeded
RPCs of 1 byte to 64 KiB with 0 payload differences, 0 transmit decisions against SRPT outside
the FIFO share, granted but unreceived bytes never above `window`, expected bytes never above
`max_incoming`, 100 of 100 oversized requests failed `TooLong` with 0 restarts, 0 byte
differences between an answer reserved and committed and the same answer replied whole, 100 of
100 reservations dropped without a commit returning their pages, and 0 panics.

The exit number of the Network half, stage H1: the exit number of the compact half on the Network
profile in both scheduling modes, the longest probe of the table at 4,096 peers printed, the 4
source ports of one client seen by the server as 4 peers with their RPCs apart, 1,000 of 1,000
packets to a port the server does not own dropped and counted in `Stats::wrong_port`, and 100 of
100 requests from a port outside the client's own failed `SendError::Port`.

The exit number of the real socket half, stage H2: 0 payload differences in both directions and
both forms over 10,000 seeded RPCs of 1 byte to 64 KiB on each form in each direction, 10 of 10
small answers complete before the 64 KiB answer on each form, 1 truncated record of 16,385 bytes
dropped and counted, 100 of 100 oversized requests failed `TooLong`, 0 panics, and the table
against AoFrame alone complete for both forms.

Judged stages: H0, H1 and H2 of the Homa track.

In [ ]:
# Papermill overrides this cell.
# The tree under comparison.
aosabi_root = "."

# The half this entry runs: compact, the point to point profiles of stage H0,
# network, the Network profile of stage H1, or socket, the real Unix Sockets
# of stage H2.
half = "compact"

# The seed of the sizes and the bytes of every RPC.
seed = 20260927

# The RPCs of the exit number, shared out over the profiles below.
max_ops = 10000

# The oversized requests and the dropped reservations of the exit number.
oversized_ops = 100
reservation_ops = 100

# The RPCs a scenario carries, so one trace stays a few megabytes.
chunk = 500

# The RPCs A keeps open at once, and the RPCs B accepts open from A.
window = 4

# The RPCs of the Network half under the bound that binds its grants, per
# scheduling mode.
binding_ops = 2000

# The table of the Network half, the ports case and its counts.
table_peers = 4096
client_ports = 4
rpcs_per_port = 25
foreign_packets = 1000
outside_requests = 100

# The real socket half: the RPCs of each form, in one process and in each
# direction across two processes, the small requests sent while a 64 KiB
# answer is in flight, the send buffer of the server that keeps that answer in
# its engine, the milliseconds the client reads nothing after the small
# requests, the oversized requests of each direction and form, and the draws
# of the table against AoFrame alone.
socket_ops = 10000
small_ops = 10
serve_send_buffer = 16384
small_pause_ms = 50
socket_oversized = 100
table_draws = 20

# The per command timeout.
timeout_s = 600

# The pinned toolchain the crate is built with.
rust_version = "1.95.0"

## Setup

The cell prints the toolchain, the installed targets and the gates of the plan's section The
commands that the notebook rests on: the build of the devtool, the lints of the workspace and the
format of the crates `homa` and `aosabi-devtool`, every cargo call with `--locked`.

In [ ]:
import hashlib
import json
import math
import os
import random
import sys
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "tests" / "harness" / "__init__.py").is_file():
        sys.path.insert(0, str(_base / "tests"))
        break

from harness import paths, run
from harness.homa import fabric, trace

P = paths(aosabi_root)
OUT = P.out / "21_homa_loopback"
OUT.mkdir(parents=True, exist_ok=True)
DEVTOOL = P.target / "debug" / "aosabi-devtool"
CARGO_BIN = Path(os.environ.get("CARGO_HOME") or Path.home() / ".cargo") / "bin"
CARGO = CARGO_BIN / "cargo"
RUSTUP = CARGO_BIN / "rustup"
RUSTFMT = CARGO_BIN / "rustfmt"
CORPUS = P.tests / "capture" / "20_corpus"

# The gates of the plan's section The commands that this notebook rests on,
# each cargo call with --locked: the build of the seam it drives, the lints of
# the workspace, and the format of the crate homa under its own edition 2021
# and of the devtool, which rustfmt checks because cargo fmt takes no
# --locked.
GATES = [
    ("cargo build --locked -p aosabi-devtool",
     [CARGO, "build", "--locked", "-p", "aosabi-devtool"]),
    ("cargo clippy --locked --workspace --all-targets -- -D warnings",
     [CARGO, "clippy", "--locked", "--workspace", "--all-targets", "--", "-D", "warnings"]),
    ("rustfmt --check --edition 2021 crates/homa/src/lib.rs",
     [RUSTFMT, "--check", "--edition", "2021", P.root / "crates" / "homa" / "src" / "lib.rs"]),
    ("rustfmt --check --edition 2024 crates/aosabi-devtool/src/main.rs",
     [RUSTFMT, "--check", "--edition", "2024", P.root / "crates" / "aosabi-devtool" / "src" / "main.rs"]),
]
TARGETS = run([RUSTUP, "target", "list", "--installed"], cwd=P.root, timeout_s=60).out.split()
GATE_FAILURES = []
for label, argv in GATES:
    gate = run(argv, cwd=P.root, timeout_s=900, check=False)
    print("gate %-66s exit %d, %.1f s" % (label, gate.returncode, gate.duration_s))
    if gate.returncode != 0:
        GATE_FAILURES.append(label)
        print((gate.err or gate.out).strip()[-2000:])

reported_cargo = run([CARGO, "--version"], cwd=P.root, timeout_s=timeout_s).out.strip()
smoke = json.loads(run([DEVTOOL, "homa-sim", "--smoke"], cwd=P.root, timeout_s=timeout_s).out)
print("repository root:", P.rel(P.root))
print("devtool        :", P.rel(DEVTOOL))
print("cargo          :", reported_cargo)
print("targets        :", ", ".join(TARGETS))
print("seam           :", json.dumps(smoke))
print("seed           :", seed)
print("half           :", half)
assert not GATE_FAILURES, GATE_FAILURES
assert rust_version in reported_cargo, reported_cargo
assert half in ("compact", "network", "socket"), half
assert smoke == {"subcommand": "homa-sim", "implemented": True, "lands_at": "H0"}, smoke

PANICS = []


def sim(requests, name):
    """Send scenarios to `homa-sim` in one process and return its answers."""
    path = OUT / ("%s.ndjson" % name)
    path.write_text("".join(json.dumps(r) + "\n" for r in requests), encoding="utf-8")
    result = run(
        [DEVTOOL, "homa-sim", "--requests", path], cwd=P.root, timeout_s=timeout_s, check=False
    )
    answers = [json.loads(line) for line in result.out.splitlines()]
    if result.returncode != 0 or len(answers) != len(requests):
        PANICS.append("%s: exit %d, %d answers to %d requests, %s" % (
            name, result.returncode, len(answers), len(requests), result.err.strip()[-300:]))
    bad = [a for a in answers if "fatal" in a or "config_error" in a]
    assert not bad, "the seam refused scenarios of %s: %s" % (name, bad[:2])
    return answers


def fill(spec, length):
    """The bytes of a message, as the devtool's scenario defines a fill."""
    if "hex" in spec:
        pattern = bytes.fromhex(spec["hex"])
        return (pattern * (length // max(len(pattern), 1) + 1))[:length] if pattern else bytes(length)
    out = bytearray()
    block = 0
    while len(out) < length:
        out += hashlib.sha256(spec["seed"].to_bytes(8, "little") + block.to_bytes(8, "little")).digest()
        block += 1
    return bytes(out[:length])


def digest(spec, length):
    return hashlib.sha256(fill(spec, length)).hexdigest()

## The profiles and the corpus

Both halves draw their sizes from the corpus below, and the compact half runs the four
sections that follow it over the point to point profiles.

Every point to point profile of the design note, first version, section 3.1 and second version,
section 3.10 runs, each built by its constructor of `homa::Profile`, with a link model of the
carrier: the rate, the packets one connection event or the queue below the engine holds, the
interval of the events and the latency.

`max_incoming` is the sum of the slot capacities the link may fill. Each end announces a
`max_message` of 65,536 bytes, the 16 pages of 4,096 bytes a message may span, and B accepts
`window` RPCs open from A, which opens at most `window` at once, so the link fills at most four
slots of each end and the bound of the design is 262,144 bytes on every profile. The runs under that
bound hold the expected bytes to it at every row of the trace. Since no receiver of these runs
expects that much, the grants never meet it, so the same RPCs run a second time under a bound that
binds them, the unscheduled bytes of the RPCs B accepts open plus half a window, and there the bound
holds at every GRANT a receiver sends.

The sizes are drawn with the seed: a third of the requests take the length of a request body of the
committed corpora of `tests/capture/20_corpus/`, the operations the executor notebook sends, since
the Homa layer carries bytes of those lengths, and the rest and every answer take a length from 1
byte to 64 KiB, uniform in its logarithm. One RPC in five carries Japanese text in UTF-8 as its
bytes, and every other RPC carries the SHA-256 blocks of its seed.

In [ ]:
PROFILES = [
    ("unix_socket", {"name": "unix_socket", "max_message": 65536},
     {"bytes_per_s": 0, "packets_per_event": 2, "event_ns": 0, "latency_ns": 10_000}, 0.30),
    ("usb_bulk_high_speed", {"name": "usb_bulk", "high_speed": True},
     {"bytes_per_s": 40_000_000, "packets_per_event": 2, "event_ns": 0, "latency_ns": 125_000}, 0.20),
    ("usb_bulk_full_speed", {"name": "usb_bulk", "high_speed": False},
     {"bytes_per_s": 1_000_000, "packets_per_event": 2, "event_ns": 0, "latency_ns": 1_000_000}, 0.15),
    ("usb_serial", {"name": "usb_serial"},
     {"bytes_per_s": 1_000_000, "packets_per_event": 2, "event_ns": 0, "latency_ns": 1_000_000}, 0.10),
    ("bluetooth_channel", {"name": "bluetooth_channel", "peer_mps": 247, "interval_ns": 30_000_000},
     {"bytes_per_s": 36_000, "packets_per_event": 2, "event_ns": 30_000_000, "latency_ns": 1_000_000}, 0.10),
    ("bluetooth_gatt", {"name": "bluetooth_gatt", "att_mtu": 185, "interval_ns": 30_000_000},
     {"bytes_per_s": 12_133, "packets_per_event": 2, "event_ns": 30_000_000, "latency_ns": 1_000_000}, 0.08),
    ("serial_line", {"name": "serial_line", "baud": 115200},
     {"bytes_per_s": 11_520, "packets_per_event": 1, "event_ns": 0, "latency_ns": 100_000}, 0.07),
]
CONFIG = {"client_slots": 8, "server_slots": 8, "page_len": 4096, "pages": 256, "max_gaps": 2}

# The bytes one slot holds, the max_message both ends announce, and the
# bound of the design note, first version, section 3.1, over the slots the
# link may fill.
SLOT_CAPACITY = 65536

values = {}
for name, spec, model, _ in PROFILES:
    answer = sim([{"id": 0, "op": "profile", "profile": spec}], "profile-" + name)[0]["profile"]
    values[name] = answer
print("%-22s %-10s %-12s %-10s %-12s %-12s %-12s %-12s" % (
    "profile", "packet", "unscheduled", "window", "capacity", "binding", "resend ms", "fifo_every"))
for name, spec, model, _ in PROFILES:
    v = values[name]
    v["max_incoming_capacity"] = window * SLOT_CAPACITY
    v["max_incoming_run"] = window * v["unsched_bytes"] + v["window"] // 2
    print("%-22s %-10d %-12d %-10d %-12d %-12d %-12d %-12d" % (
        name, v["max_packet"], v["unsched_bytes"], v["window"], v["max_incoming_capacity"],
        v["max_incoming_run"], v["timers"]["resend_after_ns"] // 1_000_000, v["fifo_every"]))

request_bodies = []
for path in sorted(CORPUS.glob("*.json")):
    if path.name == "manifest.json":
        continue
    for case in json.loads(path.read_text(encoding="utf-8"))["cases"]:
        body = {"query": case["document"]}
        if case.get("variables") is not None:
            body["variables"] = case["variables"]
        request_bodies.append(len(json.dumps(body, ensure_ascii=False).encode()))
print()
print("request bodies of the corpora:", len(request_bodies),
      "from", min(request_bodies), "to", max(request_bodies), "bytes")

JAPANESE = [
    "日本語のテキスト、漢字とひらがなとカタカナ。".encode(),
    "{\"data\":{\"user\":{\"name\":\"山田太郎\",\"city\":\"東京都\"}}}".encode(),
    "こんにちは、世界。今日はいい天気ですね。".encode(),
]
g = random.Random(seed)


def log_size():
    return max(1, min(65536, int(round(2 ** g.uniform(0, 16)))))


def draw(index):
    length = g.choice(request_bodies) if g.random() < 1 / 3 else log_size()
    response = log_size()
    if g.random() < 0.2:
        text = g.choice(JAPANESE).hex()
        return {"len": length, "fill": {"hex": text}, "response_len": response, "response_fill": {"hex": text}}
    return {"len": length, "fill": {"seed": seed * 100_000 + 2 * index},
            "response_len": response, "response_fill": {"seed": seed * 100_000 + 2 * index + 1}}


PLAN = []
index = 0
for name, spec, model, share in PROFILES:
    count = round(max_ops * share)
    rpcs = [draw(index + i) for i in range(count)]
    index += count
    PLAN.append((name, spec, model, rpcs))
total = sum(len(rpcs) for _, _, _, rpcs in PLAN)
japanese = sum(1 for _, _, _, rpcs in PLAN for r in rpcs if "hex" in r["fill"])
print("RPCs:", total, "Japanese:", japanese, "by profile:", {name: len(rpcs) for name, _, _, rpcs in PLAN})
assert total == max_ops, total

## 10,000 RPCs over the loopback

Every profile runs its RPCs in scenarios of `chunk` RPCs, one devtool process for all of them, with
half of the answers through `reply_reserve` and half through `reply`. For every RPC the cell checks
the outcome, the SHA-256 of the request B read and of the answer A read against the bytes it asked
for, and that each end reported the RPC once. The trace of each scenario goes through `trace.py`.
The 10,000 RPCs run twice, once under each bound of the section above.

The window holds at every packet a receiver takes and every GRANT it sends, in both runs. Under the
bound of the design note, the bytes a receiver expects, unscheduled or granted and not received,
stay within `max_incoming` at every row of the trace, which is the exit number as the plan states
it. Under the binding bound they stay within it at every GRANT, since a receiver grants only while
they do, and the rows at which the unscheduled bytes of a message that starts took a receiver above
that bound are printed beside, since no decision of the receiver sends them.

Every GRANT goes to the one message the scheduler of the receiver names at a decision, the message
of its sender with the fewest bytes not yet received, ties to the older, or the oldest on a turn of
the FIFO share, which is the plan's check of at most one granted message per sender, read per
decision. A receiver decides after every DATA packet it takes and sends the GRANT when its link is
ready, so `trace.py` looks for each GRANT for a DATA packet the receiver took since the last GRANT
of that message at which SRPT or the FIFO share named the message, and counts every GRANT without
one against the order. The trace does not mark the turns of the share, so the GRANTs to the oldest
message where SRPT named another are held to the share by their count, at most one in `fifo_every`
of the GRANTs a receiving side sent plus one, as the transmit decisions are. A trace the cell
builds, whose receiver grants the older and longer message at every decision, shows that the count
refuses a scheduler that the order alone lets through, since the FIFO share names the oldest message
at every decision. The most messages of one sender that held granted bytes not yet received beyond
their unscheduled bytes when a GRANT left is printed beside, which SRPT raises above one when a
shorter message takes the grants while an older one still waits for bytes it was granted.

In [ ]:
if half == 'compact':
    BOUNDS = (
        ("capacity", "max_incoming_capacity"),
        ("binding", "max_incoming_run"),
    )
    RUNS = {}
    for bound, key in BOUNDS:
        scenarios = []
        for name, spec, model, rpcs in PLAN:
            for start in range(0, len(rpcs), chunk):
                scenarios.append({
                    "id": len(scenarios),
                    "op": "run",
                    "link": "loopback",
                    "profile": spec,
                    "overrides": {"max_incoming": values[name][key]},
                    "mode": "prefix",
                    "model": model,
                    "config": CONFIG,
                    "limits_a": {"max_message": SLOT_CAPACITY},
                    "limits_b": {"max_rpcs": window, "max_message": SLOT_CAPACITY},
                    "window": window,
                    "reserve": "half",
                    "seed": seed,
                    "trace": str(OUT / ("trace-%s-%s-%d.ndjson" % (bound, name, start))),
                    "rpcs": rpcs[start:start + chunk],
                    "_profile": name,
                })
        answers = sim([{k: v for k, v in s.items() if k != "_profile"} for s in scenarios], "loopback-" + bound)

        results = {"payload_differences": 0, "not_ok": 0, "duplicates": 0, "reserved": 0, "replied": 0}
        order_counts = {"decisions": 0, "fifo": 0, "against": 0, "share_exceeded": 0}
        grants = {"window_over": 0, "incoming_over": 0, "expected_over_any_row": 0}
        grant_counts = {"grants": 0, "by_srpt": 0, "by_fifo": 0, "against": 0, "not_judged": 0,
                        "fifo_share": 0, "share_exceeded": 0}
        per_profile = {}
        for scenario, answer in zip(scenarios, answers):
            name = scenario["_profile"]
            row = per_profile.setdefault(name, {"rpcs": 0, "sim_s": 0.0, "decisions": 0, "against": 0,
                                                "worst_window": 0, "worst_incoming": 0, "per_sender": 0,
                                                "stopped": set()})
            row["rpcs"] += len(scenario["rpcs"])
            row["sim_s"] += answer["end_ns"] / 1e9
            row["stopped"].add(answer["stopped"])
            for n, (spec, got) in enumerate(zip(scenario["rpcs"], answer["rpcs"])):
                if got["outcome"] != "ok":
                    results["not_ok"] += 1
                    continue
                if got["request_sha"] != digest(spec["fill"], spec["len"]) or \
                        got["response_sha"] != digest(spec["response_fill"], spec["response_len"]):
                    results["payload_differences"] += 1
                if got["server_requests"] != 1 or got["client_responses"] != 1:
                    results["duplicates"] += 1
                results["reserved" if n % 2 == 1 else "replied"] += 1
            rows = trace.read(scenario["trace"])
            order = trace.srpt(rows, values[name]["fifo_every"])
            for k in ("decisions", "fifo", "against"):
                order_counts[k] += order[k]
            order_counts["share_exceeded"] += len(order["share_exceeded"])
            row["decisions"] += order["decisions"]
            row["against"] += order["against"]
            v = values[name]
            unsched = {"A": v["unsched_bytes"], "B": v["unsched_bytes"]}
            held = trace.grants(
                rows,
                unsched=unsched,
                window={"A": v["window"], "B": v["window"]},
                max_incoming={"A": v[key], "B": v[key]},
            )
            grants["window_over"] += held["window_over"]
            grants["incoming_over"] += held["incoming_over"]
            grants["expected_over_any_row"] += held["expected_over_any_row"]
            row["worst_window"] = max(row["worst_window"], *held["worst_window"].values())
            row["worst_incoming"] = max(row["worst_incoming"], *held["worst_incoming"].values())
            row["per_sender"] = max(row["per_sender"], *trace.granted_per_sender(rows, unsched).values())
            chosen = trace.grant_order(rows, unsched, {"A": v["fifo_every"], "B": v["fifo_every"]})
            for k in ("grants", "by_srpt", "by_fifo", "against", "not_judged", "fifo_share"):
                grant_counts[k] += chosen[k]
            grant_counts["share_exceeded"] += len(chosen["share_exceeded"])
            if order["against"] or held["window_over"] or held["incoming_over"] or chosen["against"] or \
                    chosen["share_exceeded"]:
                print("  %s: %s %s %s %s %s" % (name, order["examples"][:2], held["window_examples"][:2],
                                                held["incoming_examples"][:2], chosen["examples"][:2],
                                                chosen["share_exceeded"]))
            if bound == "capacity" and held["expected_over_any_row"]:
                print("  %s: %s" % (name, held["expected_over_examples"][:2]))
            Path(scenario["trace"]).unlink()
        RUNS[bound] = {"results": results, "srpt": order_counts, "grants": grants, "grant_order": grant_counts}

        print("under the %s bound of max_incoming:" % bound)
        print("%-22s %-6s %-12s %-10s %-8s %-11s %-15s %-11s %s" % (
            "profile", "RPCs", "simulated s", "decisions", "against", "worst held", "worst expected",
            "per sender", "stopped"))
        for name, row in per_profile.items():
            print("%-22s %-6d %-12.1f %-10d %-8d %-11d %-15d %-11d %s" % (
                name, row["rpcs"], row["sim_s"], row["decisions"], row["against"], row["worst_window"],
                row["worst_incoming"], row["per_sender"], ",".join(sorted(row["stopped"]))))
        print("RPCs:", total, results)
        print("transmit decisions:", order_counts)
        print("grants:", grants)
        print("grant order:", grant_counts)
        print()

    # A receiver B that grants the older and longer message 1 at each of 20
    # decisions, while SRPT names the shorter message 2 at every one of them.
    # The FIFO share names message 1 at each decision, so the order alone lets
    # the GRANTs through, and the count of the share of 1 in 16 refuses them.
    FIFO_ONLY = [["R", 0, "B", 1, 0, 100, 5000], ["R", 1, "B", 2, 0, 100, 1000]]
    for n in range(20):
        FIFO_ONLY += [["R", 2 + 2 * n, "B", 2, 100 + n, 1, 1000], ["G", 3 + 2 * n, "B", 1, 200 + 100 * n]]
    fifo_only = trace.grant_order(FIFO_ONLY, {"A": 100, "B": 100}, {"A": 16, "B": 16})
    print("a receiver that grants the oldest at every decision:", {
        k: fifo_only[k] for k in ("grants", "by_srpt", "by_fifo", "against", "share_exceeded")})

## Oversized requests

B accepts messages of 1,000 bytes on the Serial Line profile, and A takes 64 KiB of B until B's
LIMITS arrive, so the requests A sends before them reach B above its bound. B refuses such a packet
and answers with its LIMITS, A fails every request longer than B's `max_message` with
`Failure::TooLong`, and every later request of that length is refused `SendError::TooLong` at once,
and no request restarts.

In [ ]:
if half == 'compact':
    over = random.Random(seed ^ 0x21)
    oversized = [{"len": over.randint(1001, 8000), "response_len": 10} for _ in range(oversized_ops)]
    answer = sim([{
        "id": 0, "op": "run", "link": "loopback",
        "profile": {"name": "serial_line", "baud": 115200},
        "model": {"bytes_per_s": 11_520, "packets_per_event": 1, "latency_ns": 100_000},
        "config": CONFIG, "window": window,
        "limits_b": {"max_message": 1000}, "assumed_a": {"max_message": 65536},
        "rpcs": oversized,
    }], "oversized")[0]
    too_long = {"failed": 0, "refused": 0}
    other = []
    for got in answer["rpcs"]:
        if got["why"] == "TooLong" and got["outcome"] in too_long:
            too_long[got["outcome"]] += 1
        else:
            other.append((got["outcome"], got["why"]))
    oversized_failed = too_long["failed"] + too_long["refused"]
    oversized_restarts = answer["stats"]["A"]["restarts"]
    print("oversized requests failed TooLong: %d of %d, %s" % (oversized_failed, oversized_ops, too_long))
    print("other outcomes:", other[:5], " restarts at A:", oversized_restarts,
          " packets B refused:", answer["stats"]["B"]["refused"])

## Reserved answers against whole answers

The same 200 RPCs run twice on the USB serial profile under one seed, once with every answer
reserved, written in chunks and committed, and once with every answer replied whole. The packets
each end put on the link are hashed in order by the devtool, and both runs must put the same
bytes on the link, so an answer reserved and committed differs from the same answer replied whole
in no byte, and each answer A read must be the answer B was asked for.

In [ ]:
if half == 'compact':
    same = random.Random(seed ^ 0x22)
    pair_rpcs = [{"len": same.randint(1, 3000), "response_len": same.randint(1, 30000),
                  "reserve_max": 30000} for _ in range(200)]
    pair = sim([
        {"id": mode, "op": "run", "link": "loopback", "profile": {"name": "usb_serial"},
         "model": {"bytes_per_s": 1_000_000, "packets_per_event": 2, "latency_ns": 1_000_000},
         "config": CONFIG, "window": window, "reserve": mode, "rpcs": pair_rpcs}
        for mode in ("all", "none")
    ], "reserve-pair")
    reserved_run, whole_run = pair
    wire_equal = reserved_run["wire_sha"] == whole_run["wire_sha"]
    answers_equal = [a["response_sha"] for a in reserved_run["rpcs"]] == [a["response_sha"] for a in whole_run["rpcs"]]
    expected_answers = sum(
        1 for spec, got in zip(pair_rpcs, reserved_run["rpcs"])
        if got["response_sha"] == digest({"seed": 2 * got["index"] + 1}, spec["response_len"])
    )
    reserve_differences = (not wire_equal) + (not answers_equal) + (len(pair_rpcs) - expected_answers)
    print("wire of the reserved run:", reserved_run["wire_sha"])
    print("wire of the whole run   :", whole_run["wire_sha"])
    print("the same bytes on the link:", wire_equal, " the same answers:", answers_equal,
          " answers equal to the bytes asked for: %d of %d" % (expected_answers, len(pair_rpcs)))

## Reservations dropped without a commit

B holds one server slot and a pool of five pages of 1,024 bytes, one for a request and four for an
answer of 4,096 bytes, so a reservation of the whole answer takes every free page. Each of the 100
RPCs has B reserve the answer, write into it and drop it without a commit, and then reserve it
again, write it and commit. The second reservation takes its four pages only where the first
returned them, and B still holds the request after the drop.

In [ ]:
if half == 'compact':
    drops = [{"len": 100 + n, "response_len": 4096, "reserve": True, "reserve_max": 4096,
              "drop_reservations": 1} for n in range(reservation_ops)]
    dropped_run = sim([{
        "id": 0, "op": "run", "link": "loopback", "profile": {"name": "usb_serial"},
        "model": {"bytes_per_s": 1_000_000, "packets_per_event": 2, "latency_ns": 1_000_000},
        "config": CONFIG,
        "config_b": {"client_slots": 0, "server_slots": 1, "page_len": 1024, "pages": 5},
        "limits_a": {"max_message": 4096}, "limits_b": {"max_rpcs": 1, "max_message": 1024},
        "window": 1, "rpcs": drops,
    }], "reservations")[0]
    returned = sum(1 for got in dropped_run["rpcs"] if got["dropped_reservations"] == 1 and got["returned_reservations"] == 1 and got["kept_requests"] == 1)
    dropped_ok = sum(1 for got in dropped_run["rpcs"] if got["outcome"] == "ok")
    print("reservations dropped:", sum(got["dropped_reservations"] for got in dropped_run["rpcs"]))
    print("dropped reservations whose pages the next reservation took, the request kept: %d of %d" % (returned, reservation_ops))
    print("RPCs answered whole after the drop: %d of %d" % (dropped_ok, reservation_ops))

## The Network half: RPCs in both scheduling modes

The Network half runs the Network profile of `homa::Profile::network` with the full form, both ends
at the MTU of 1,500 over IPv4, on the loopback of `homa::sim` with the model of a 25 Gbit/s link
that holds two packets below the engine and delivers after 1.5 microseconds. The profile's own
`max_incoming` is 400,000 bytes, its window the dynamic one of HomaModule, `max_incoming` divided by
one more than the messages granted at once, and its FIFO share the interval of HomaModule, a grant
of `fifo_grant_increment` bytes to the oldest message every time that many bytes take 5 percent of
the link.

The `max_ops` RPCs of the corpus above run once in the prefix mode and once under all or nothing,
where a message longer than its 40,000 unscheduled bytes opens with `START_MSG` and waits for a
GRANT. For every RPC the cell checks the outcome and the SHA-256 of both messages, and the trace
goes through `trace.py`: SRPT order of the transmit decisions, where Network has no FIFO share,
every GRANT to the message the scheduler names with the grants to the oldest held to the decisions
of the FIFO share the `P` rows mark, and each decision of the grant scheduler: its message granted
at most the window it names beyond what arrived, that window the dynamic one, its level that of
`homa_grant_priority`, and the bytes the receiver expects within `max_incoming` with the grant
included. Under the profile's own bound the bytes a receiver expects stay within it at every row of
the trace. `binding_ops` RPCs of each mode then run under a bound of twice the unscheduled bytes,
80,000, which binds the grants, and there the decisions hold it.

A grant of the FIFO share adds `fifo_grant_increment` bytes whatever the window, as HomaModule's
does, so the window is checked at the decisions outside it, and the decisions of the share are
counted beside.

In [ ]:
if half == "network":
    NET_SPEC = {"name": "network"}
    net = sim([{"id": 0, "op": "profile", "profile": NET_SPEC}], "profile-network")[0]["profile"]
    NET_MODEL = {"bytes_per_s": 3_125_000_000, "packets_per_event": 2, "event_ns": 0, "latency_ns": 1_500}
    NET_CONFIG = {"client_slots": 8, "server_slots": 8, "page_len": 4096, "pages": 256, "max_gaps": 16}
    NET_UNSCHED = net["unsched_bytes"]
    NET_BOUNDS = (("profile", net["max_incoming"], max_ops), ("binding", 2 * NET_UNSCHED, binding_ops))
    print("Network: packet %d, unscheduled %d, max_incoming %d, fifo_every %d, scheduling %s" % (
        net["max_packet"], NET_UNSCHED, net["max_incoming"], net["fifo_every"], net.get("scheduling")))
    gn = random.Random(seed ^ 0x2101)

    def net_size():
        return max(1, min(65536, int(round(2 ** gn.uniform(0, 16)))))

    def net_draw(index):
        length = gn.choice(request_bodies) if gn.random() < 1 / 3 else net_size()
        response = net_size()
        if gn.random() < 0.2:
            text = gn.choice(JAPANESE).hex()
            return {"len": length, "fill": {"hex": text}, "response_len": response, "response_fill": {"hex": text}}
        return {"len": length, "fill": {"seed": seed * 100_000 + 50_000 + 2 * index},
                "response_len": response, "response_fill": {"seed": seed * 100_000 + 50_000 + 2 * index + 1}}

    NET_RPCS = [net_draw(i) for i in range(max_ops)]
    NET_RUNS = {}
    for bound, max_incoming, count in NET_BOUNDS:
        for mode in ("prefix", "all-or-nothing"):
            rpcs = NET_RPCS[:count]
            scenarios = []
            for start in range(0, len(rpcs), chunk):
                scenarios.append({
                    "id": len(scenarios), "op": "run", "link": "loopback", "form": "full",
                    "profile": NET_SPEC, "overrides": {"max_incoming": max_incoming}, "mode": mode,
                    "model": NET_MODEL, "config": NET_CONFIG,
                    "limits_a": {"max_message": SLOT_CAPACITY},
                    "limits_b": {"max_rpcs": window, "max_message": SLOT_CAPACITY},
                    "window": window, "reserve": "half", "seed": seed,
                    "trace": str(OUT / ("trace-network-%s-%s-%d.ndjson" % (bound, mode, start))),
                    "rpcs": rpcs[start:start + chunk],
                })
            answers = sim(scenarios, "network-%s-%s" % (bound, mode))
            aon = mode == "all-or-nothing"
            unsched = {"A": NET_UNSCHED, "B": NET_UNSCHED}
            bounds = {"A": max_incoming, "B": max_incoming}
            got = {"rpcs": 0, "payload_differences": 0, "not_ok": 0, "duplicates": 0, "decisions_tx": 0,
                   "against": 0, "share_exceeded": 0, "expected_over_any_row": 0, "grants": 0,
                   "grant_against": 0, "grant_share_exceeded": 0, "decisions": 0, "fifo": 0,
                   "window_over": 0, "window_formula": 0, "level_mismatch": 0, "expected_over": 0,
                   "worst_beyond": 0, "worst_expected": 0, "per_sender": 0, "start_msg": 0, "sim_s": 0.0}
            stopped = set()
            for scenario, answer in zip(scenarios, answers):
                stopped.add(answer["stopped"])
                got["sim_s"] += answer["end_ns"] / 1e9
                got["start_msg"] += answer["stats"]["A"]["start_msg_sent"] + answer["stats"]["B"]["start_msg_sent"]
                for spec, rpc in zip(scenario["rpcs"], answer["rpcs"]):
                    got["rpcs"] += 1
                    if rpc["outcome"] != "ok":
                        got["not_ok"] += 1
                        continue
                    if rpc["request_sha"] != digest(spec["fill"], spec["len"]) or \
                            rpc["response_sha"] != digest(spec["response_fill"], spec["response_len"]):
                        got["payload_differences"] += 1
                    if rpc["server_requests"] != 1 or rpc["client_responses"] != 1:
                        got["duplicates"] += 1
                rows = trace.read(scenario["trace"])
                order = trace.srpt(rows, 0)
                got["decisions_tx"] += order["decisions"]
                got["against"] += order["against"]
                got["share_exceeded"] += len(order["share_exceeded"])
                held = trace.grants(rows, unsched, {"A": max_incoming, "B": max_incoming}, bounds,
                                    all_or_nothing=aon)
                got["expected_over_any_row"] += held["expected_over_any_row"]
                share = trace.decision_share(rows)
                chosen = trace.grant_order(rows, unsched, {"A": 0, "B": 0}, all_or_nothing=aon,
                                           fifo_allowed=share)
                got["grants"] += chosen["grants"]
                got["grant_against"] += chosen["against"] + chosen["not_judged"]
                got["grant_share_exceeded"] += len(chosen["share_exceeded"])
                made = trace.decisions(rows, unsched, bounds, 8, fabric.homa_grant_priority,
                                       all_or_nothing=aon)
                for k in ("decisions", "fifo", "window_over", "window_formula", "level_mismatch", "expected_over"):
                    got[k] += made[k]
                got["worst_beyond"] = max(got["worst_beyond"], made["worst_beyond"])
                got["worst_expected"] = max(got["worst_expected"], *made["worst_expected"].values())
                got["per_sender"] = max(got["per_sender"], *trace.granted_per_sender(rows, unsched, aon).values())
                if order["against"] or chosen["against"] or made["window_over"] or made["expected_over"] or \
                        made["level_mismatch"] or made["window_formula"]:
                    print("  %s %s: %s %s %s %s" % (bound, mode, order["examples"][:2], chosen["examples"][:2],
                                                   made["window_examples"][:2], made["expected_examples"][:2]))
                Path(scenario["trace"]).unlink()
            got["stopped"] = sorted(stopped)
            NET_RUNS[(bound, mode)] = got
    print()
    print("%-9s %-15s %-6s %-9s %-10s %-8s %-7s %-10s %-6s %-12s %-14s %-9s %s" % (
        "bound", "mode", "RPCs", "payload", "transmit", "against", "GRANTs", "decisions", "FIFO",
        "worst beyond", "worst expected", "START_MSG", "stopped"))
    for (bound, mode), got in NET_RUNS.items():
        print("%-9s %-15s %-6d %-9d %-10d %-8d %-7d %-10d %-6d %-12d %-14d %-9d %s" % (
            bound, mode, got["rpcs"], got["payload_differences"], got["decisions_tx"], got["against"],
            got["grants"], got["decisions"], got["fifo"], got["worst_beyond"], got["worst_expected"],
            got["start_msg"], ",".join(got["stopped"])))
    for key, got in NET_RUNS.items():
        print(key, {k: v for k, v in got.items() if k not in ("stopped",)})

## The Network half: oversized requests, reserved answers and dropped reservations

The full form carries no LIMITS, so a client knows the `max_message` of its peer from its own
configuration, as every host of HomaModule knows `HOMA_MAX_MESSAGE_LENGTH`. A requests above the
65,536 bytes it assumes of B, and each one fails at once with `SendError::TooLong`, with no restart.
The pair of reserved and whole answers and the dropped reservations of the compact half run again on
the Network profile, the pair in both scheduling modes.

In [ ]:
if half == "network":
    over = random.Random(seed ^ 0x2102)
    net_oversized = [{"len": over.randint(SLOT_CAPACITY + 1, SLOT_CAPACITY + 14_464), "response_len": 10}
                     for _ in range(oversized_ops)]
    answer = sim([{
        "id": 0, "op": "run", "link": "loopback", "form": "full", "profile": NET_SPEC,
        "model": NET_MODEL, "config": NET_CONFIG, "window": window,
        "limits_b": {"max_message": SLOT_CAPACITY}, "assumed_a": {"max_message": SLOT_CAPACITY},
        "rpcs": net_oversized,
    }], "network-oversized")[0]
    net_too_long = {"failed": 0, "refused": 0}
    net_other = []
    for got in answer["rpcs"]:
        if got["why"] == "TooLong" and got["outcome"] in net_too_long:
            net_too_long[got["outcome"]] += 1
        else:
            net_other.append((got["outcome"], got["why"]))
    net_oversized_failed = net_too_long["failed"] + net_too_long["refused"]
    net_oversized_restarts = answer["stats"]["A"]["restarts"]
    print("oversized requests failed TooLong: %d of %d, %s, restarts at A: %d, other outcomes: %s" % (
        net_oversized_failed, oversized_ops, net_too_long, net_oversized_restarts, net_other[:5]))

    same = random.Random(seed ^ 0x2103)
    net_pair_rpcs = [{"len": same.randint(1, 3000), "response_len": same.randint(1, 30000),
                      "reserve_max": 30000} for _ in range(200)]
    net_reserve_differences = 0
    for mode in ("prefix", "all-or-nothing"):
        pair = sim([
            {"id": kind, "op": "run", "link": "loopback", "form": "full", "profile": NET_SPEC, "mode": mode,
             "model": NET_MODEL, "config": NET_CONFIG, "window": window, "reserve": kind, "rpcs": net_pair_rpcs}
            for kind in ("all", "none")
        ], "network-reserve-pair-" + mode)
        reserved_run, whole_run = pair
        wire_equal = reserved_run["wire_sha"] == whole_run["wire_sha"]
        answers_equal = [a["response_sha"] for a in reserved_run["rpcs"]] == [a["response_sha"] for a in whole_run["rpcs"]]
        expected_answers = sum(
            1 for spec, got in zip(net_pair_rpcs, reserved_run["rpcs"])
            if got["response_sha"] == digest({"seed": 2 * got["index"] + 1}, spec["response_len"])
        )
        net_reserve_differences += (not wire_equal) + (not answers_equal) + (len(net_pair_rpcs) - expected_answers)
        print("%s: the same bytes on the link: %s, the same answers: %s, answers equal to the bytes asked for: %d of %d" % (
            mode, wire_equal, answers_equal, expected_answers, len(net_pair_rpcs)))

    net_drops = [{"len": 100 + n, "response_len": 4096, "reserve": True, "reserve_max": 4096,
                  "drop_reservations": 1} for n in range(reservation_ops)]
    net_dropped_run = sim([{
        "id": 0, "op": "run", "link": "loopback", "form": "full", "profile": NET_SPEC,
        "model": NET_MODEL, "config": NET_CONFIG,
        "config_b": {"client_slots": 0, "server_slots": 1, "page_len": 1024, "pages": 5},
        "limits_a": {"max_message": 4096}, "limits_b": {"max_rpcs": 1, "max_message": 1024},
        "window": 1, "rpcs": net_drops,
    }], "network-reservations")[0]
    net_returned = sum(1 for got in net_dropped_run["rpcs"] if got["dropped_reservations"] == 1
                       and got["returned_reservations"] == 1 and got["kept_requests"] == 1)
    net_dropped_ok = sum(1 for got in net_dropped_run["rpcs"] if got["outcome"] == "ok")
    print("dropped reservations whose pages the next reservation took, the request kept: %d of %d" % (
        net_returned, reservation_ops))
    print("RPCs answered whole after the drop: %d of %d" % (net_dropped_ok, reservation_ops))

## The Network half: the ports of a peer and the table

A peer on Network is an address with its Homa port, so a client endpoint that owns four ports,
`Config::port` and three `extra_ports`, and sends `rpcs_per_port` requests from each is four peers
to the server, each with its own RPCs, and every answer returns to the port its request left from.
`foreign_packets` DATA packets for a port the server does not own are dropped and counted in
`Stats::wrong_port`, and `outside_requests` requests that name a source port the client does not own
fail `SendError::Port`. The case of the review of HB02 runs beside under `Scope::Shared`: two
requests from two ports of one client share the server's downlink with one message granted at a
time, the link of the first port goes down after its first packets, and the second must still
complete, since no packet of the first will run the scheduler again. The server takes the
acknowledgement of an RPC from any port of the client's host, as HomaModule finds it by the address
and the identifier alone.

The cell prints the longest probe of any lookup, their mean, and the probes an endpoint of the same
size counted while it admitted every peer from its first packet.

In [ ]:
if half == "network":
    ports_answer = sim([{
        "id": 0, "op": "ports", "profile": NET_SPEC, "client_ports": client_ports,
        "rpcs_per_port": rpcs_per_port, "foreign_packets": foreign_packets,
        "outside_requests": outside_requests, "down_case": True,
    }], "network-ports")[0]
    seen = ports_answer["server_peers"]
    id_sets = [set(p["ids"]) for p in seen]
    apart = all(not (a & b) for i, a in enumerate(id_sets) for b in id_sets[i + 1:])
    ports_seen = sorted(p["port"] for p in seen)
    per_peer = [len(p["ids"]) for p in seen]
    answered_by_port = {r["port"]: (r["answered"], r["equal"]) for r in ports_answer["responses"]}
    outside_port = sum(1 for o in ports_answer["outside"] if o == "Port")
    print("peers the server saw:", ports_seen, " RPCs of each:", per_peer, " apart:", apart)
    print("responses by source port, answered and equal to the request:", answered_by_port)
    print("packets to a foreign port counted in wrong_port: %d of %d, the server's counter %d" % (
        ports_answer["foreign_counted"], foreign_packets, ports_answer["server_stats"]["wrong_port"]))
    print("requests from a port outside the client's own failed Port: %d of %d" % (outside_port, outside_requests))
    print("restarts at the client:", ports_answer["client_stats"]["restarts"],
          " packets the server refused:", ports_answer["server_stats"]["refused"])
    print("the case of a shared downlink whose granted message ends with its peer:", ports_answer["down_case"])

    table = sim([{"id": 0, "op": "table", "peers": table_peers}], "network-table")[0]
    print()
    print("table at %d peers in %d buckets: longest probe %d beyond the first, mean %.3f, sum %d" % (
        table["peers"], table["buckets"], table["longest_probe"], table["mean_probe_milli"] / 1000,
        table["probes_sum"]))
    print("an endpoint of the same size admitted %d peers from their first packets with %d probes beyond the first" % (
        table["admitted"], table["endpoint_probes_beyond_first"]))

## The real socket half: 10,000 RPCs each way over real Unix Sockets

The half runs the engine of the crate over the Unix Socket transports of `homa::unix` with the Unix
Socket profile, packets of 16,384 bytes and the whole message of at most 64 KiB unscheduled, in both
forms: a `SOCK_SEQPACKET` socket whose every record is one packet, and a stream socket in prefixed
AoFrame. The sizes and the bytes are drawn as in the section of the profiles, a third from the
request bodies of the committed corpora, one RPC in five in Japanese text.

First both ends run in one process through `aosabi-devtool homa-sim` over `homa::unix::pair`, the
link `seqpacket`, and over a connection a `homa::unix::StreamListener` accepted, the link `stream`,
`socket_ops` RPCs on each. Then the ends run in two processes, the engine of the crate against the
Python engine `tests/harness/homa/client.py`, which shares no code with the crate: `aosabi-devtool
homa-unix serve` answers the requests of a client of `client.py` with their own bytes, and
`aosabi-devtool homa-unix client` sends its requests to a server of `client.py`, which answers each
with its own bytes, `socket_ops` RPCs on each form in each direction. Across two processes a
sequenced packet socket is a listener at a path and one connection, since `serve` takes a path, as
the plan's section The devtool gives it. For every RPC the cell checks the outcome and the bytes of
both messages against the bytes it asked for, and the server counts every request delivered twice.


In [ ]:
SOCKET_RESULTS = {}
if half == "socket":
    import subprocess
    import threading
    import time
    from harness.homa import client as hc

    SOCKS = OUT / "sock"
    SOCKS.mkdir(parents=True, exist_ok=True)
    UNIX_SPEC = {"name": "unix_socket", "max_message": SLOT_CAPACITY}
    UNIX_LIMITS = {"max_packet": 16384, "max_message": SLOT_CAPACITY, "unsched_bytes": SLOT_CAPACITY,
                   "max_rpcs": 16}
    unix_smoke = json.loads(run([DEVTOOL, "homa-unix", "--smoke"], cwd=P.root, timeout_s=timeout_s).out)
    print("seam homa-unix :", json.dumps(unix_smoke))
    assert unix_smoke == {"subcommand": "homa-unix", "implemented": True, "lands_at": "H2"}, unix_smoke

    def unix(requests, name, check=True):
        """Send requests to `homa-unix` in one process and return its answers."""
        path = OUT / ("%s.ndjson" % name)
        path.write_text("".join(json.dumps(r) + "\n" for r in requests), encoding="utf-8")
        result = run([DEVTOOL, "homa-unix", "--requests", path], cwd=P.root, timeout_s=timeout_s, check=False)
        answers = [json.loads(line) for line in result.out.splitlines()]
        if result.returncode != 0 or len(answers) != len(requests):
            PANICS.append("%s: exit %d, %d answers to %d requests, %s" % (
                name, result.returncode, len(answers), len(requests), result.err.strip()[-300:]))
        if check:
            bad = [a for a in answers if "fatal" in a]
            assert not bad, "the seam refused %s: %s" % (name, bad[:2])
        return answers

    def serve(request, name):
        """Start `homa-unix serve` in the background and wait for its socket."""
        sock = Path(request["path"])
        if sock.exists():
            sock.unlink()
        path = OUT / ("%s.ndjson" % name)
        path.write_text(json.dumps(dict(request, id=0, op="serve")) + "\n", encoding="utf-8")
        proc = subprocess.Popen([str(DEVTOOL), "homa-unix", "--requests", str(path)], cwd=P.root,
                                stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        hc.wait_for(sock)
        return proc

    def served(proc, name):
        """Wait for a server of `serve` and return its answer."""
        out, err = proc.communicate(timeout=timeout_s)
        answers = [json.loads(line) for line in out.splitlines()]
        if proc.returncode != 0 or len(answers) != 1 or "fatal" in answers[0]:
            PANICS.append("%s: exit %d, %s, %s" % (name, proc.returncode, out.strip()[-300:], err.strip()[-300:]))
            return {"connections": []}
        return answers[0]

    def py_client(form, path, hook=None, assumed=None, profile=None):
        """Connect a client of client.py to the server at `path`."""
        profile = profile or hc.unix_socket(SLOT_CAPACITY)
        transport = hc.open_transport(form, path, profile.max_packet, hook)
        return hc.Engine(transport, profile, hc.limits(profile, SLOT_CAPACITY), assumed or UNIX_LIMITS)

    gs = random.Random(seed ^ 0x2130)

    def socket_draw(index, salt):
        length = gs.choice(request_bodies) if gs.random() < 1 / 3 else max(1, min(65536, int(round(2 ** gs.uniform(0, 16)))))
        if gs.random() < 0.2:
            return {"len": length, "fill": {"hex": gs.choice(JAPANESE).hex()}}
        return {"len": length, "fill": {"seed": seed * 100_000 + salt + index}}

    # One process: homa-sim over homa::unix::pair and over a stream a
    # StreamListener accepted, socket_ops RPCs on each link.
    for link in ("seqpacket", "stream"):
        specs = []
        for index in range(socket_ops):
            spec = socket_draw(index, 60_000 if link == "seqpacket" else 70_000)
            answer_spec = socket_draw(index, 80_000 if link == "seqpacket" else 90_000)
            specs.append({"len": spec["len"], "fill": spec["fill"], "response_len": answer_spec["len"],
                          "response_fill": answer_spec["fill"]})
        scenarios = [{
            "id": n, "op": "run", "link": link, "profile": UNIX_SPEC, "mode": "prefix", "config": CONFIG,
            "limits_a": {"max_message": SLOT_CAPACITY}, "limits_b": {"max_rpcs": window, "max_message": SLOT_CAPACITY},
            "window": window, "reserve": "half", "seed": seed,
            "path": str(SOCKS / ("pair-%d.sock" % n)),
            "rpcs": specs[start:start + chunk],
        } for n, start in enumerate(range(0, len(specs), chunk))]
        answers = sim(scenarios, "socket-pair-" + link)
        got = {"rpcs": 0, "payload_differences": 0, "not_ok": 0, "duplicates": 0, "stopped": set()}
        for scenario, answer in zip(scenarios, answers):
            got["stopped"].add(answer["stopped"])
            for spec, rpc in zip(scenario["rpcs"], answer["rpcs"]):
                got["rpcs"] += 1
                if rpc["outcome"] != "ok":
                    got["not_ok"] += 1
                    continue
                if rpc["request_sha"] != digest(spec["fill"], spec["len"]) or \
                        rpc["response_sha"] != digest(spec["response_fill"], spec["response_len"]):
                    got["payload_differences"] += 1
                if rpc["server_requests"] != 1 or rpc["client_responses"] != 1:
                    got["duplicates"] += 1
        got["stopped"] = sorted(got["stopped"])
        SOCKET_RESULTS[("one process", link)] = got
        print("one process, %-9s %s" % (link, got))

    # Two processes: homa-unix serve answering client.py, and homa-unix
    # client driving a server of client.py, socket_ops RPCs on each form in
    # each direction.
    per_form = socket_ops
    for form in ("seqpacket", "stream"):
        salt = 100_000 if form == "seqpacket" else 110_000
        specs = [socket_draw(index, salt) for index in range(per_form)]
        messages = [fill(spec["fill"], spec["len"]) for spec in specs]
        path = str(SOCKS / ("serve-%s.sock" % form))
        proc = serve({"form": form, "path": path, "profile": UNIX_SPEC, "responder": "echo",
                      "reserve": "half", "connections": 1}, "serve-%s" % form)
        engine = py_client(form, path)
        started = time.monotonic()
        rpcs = hc.run_requests(engine, messages, window=window, timeout=timeout_s / 2)
        elapsed = time.monotonic() - started
        engine.t.sock.close()
        answer = served(proc, "serve-%s" % form)
        conn = answer["connections"][0] if answer["connections"] else {}
        got = {"rpcs": len(rpcs), "ok": sum(r.state == "ok" for r in rpcs),
               "payload_differences": sum(1 for r in rpcs if r.state == "ok" and r.response != messages[r.index]),
               "not_ok": sum(r.state != "ok" for r in rpcs), "server_duplicates": conn.get("duplicates"),
               "server_requests": conn.get("requests"), "server_answered": conn.get("answered"),
               "restarts": engine.stats["restarts"], "seconds": round(elapsed, 2)}
        SOCKET_RESULTS[("serve against client.py", form)] = got
        print("serve against client.py, %-9s %s" % (form, got))

        salt = 120_000 if form == "seqpacket" else 130_000
        specs = [socket_draw(index, salt) for index in range(per_form)]
        path = str(SOCKS / ("py-%s.sock" % form))
        server = hc.Server(form, path, hc.unix_socket(SLOT_CAPACITY), UNIX_LIMITS)
        box = []
        thread = threading.Thread(target=lambda: box.append(server.serve_one(timeout=timeout_s / 2)))
        thread.start()
        started = time.monotonic()
        answer = unix([{"id": 0, "op": "client", "form": form, "path": path, "profile": UNIX_SPEC,
                        "window": window, "requests": specs}], "client-%s" % form)[0]
        elapsed = time.monotonic() - started
        thread.join(timeout=timeout_s)
        server.close()
        py = box[0] if box else None
        got = {"rpcs": len(answer["rpcs"]), "ok": sum(r["outcome"] == "ok" for r in answer["rpcs"]),
               "payload_differences": sum(1 for spec, r in zip(specs, answer["rpcs"])
                                          if r["outcome"] == "ok" and r["sha256"] != digest(spec["fill"], spec["len"])),
               "not_ok": sum(r["outcome"] != "ok" for r in answer["rpcs"]),
               "server_requests": sum(1 for e in py.events if e[0] == "request") if py else None,
               "restarts": answer["stats"]["restarts"], "seconds": round(elapsed, 2)}
        SOCKET_RESULTS[("client against client.py", form)] = got
        print("client against client.py, %-9s %s" % (form, got))


## The real socket half: small answers, a truncated record and oversized requests

**Ten small answers behind 64 KiB.** A client of `client.py` asks `homa-unix serve` for the echo of
64 KiB, reads what arrives one packet at a time, and once the first packet of that answer has
arrived it sends `small_ops` requests of 10 bytes and reads nothing for `small_pause_ms`, so the
server reads the small requests while its socket is full. The server's socket holds
`serve_send_buffer` bytes, which the kernel doubles, and a DATA packet of the server carries at most
the rest of one of its pages of 8,192 bytes, so a few packets of the answer wait in the socket while
the rest waits in the engine, where SRPT sends every small answer before the rest of the large one.
The buffer holds a record of a whole packet as well, since a sequenced packet socket refuses a
record longer than its send buffer less 32 bytes. The cell prints the bytes of the large answer that
arrived before the first small answer and the order in which the answers completed.

**A truncated record.** On the sequenced form the client sends one record of 16,385 bytes, one
byte more than a packet, and then one RPC. The server's `SeqPacket` takes the length of the record
with `MSG_PEEK` and `MSG_TRUNC`, drops it and counts it, and the RPC after it completes. On the
stream form the same length in a prefix is refused by the decoder of AoFrame before a byte is
stored, which the cell prints beside.

**Oversized requests.** Each end assumes that its peer takes a message of 1,000,000 bytes until the
peer's LIMITS arrive, and sends `socket_oversized` requests of 65,537 to 80,000 bytes at once. The
receiver refuses the first packet of each above its 64 KiB and answers with LIMITS, and the sender
fails every such request with `TooLong`, those it had queued with `Failure::TooLong` and those after
the LIMITS with `SendError::TooLong`, and restarts none. Both directions run on both forms.


In [ ]:
if half == "socket":
    SMALL = {}
    for form in ("seqpacket", "stream"):
        path = str(SOCKS / ("small-%s.sock" % form))
        proc = serve({"form": form, "path": path, "profile": UNIX_SPEC, "responder": "echo", "reserve": "none",
                      "connections": 1, "send_buffer": serve_send_buffer}, "small-%s" % form)
        engine = py_client(form, path)
        big_bytes = random.Random(seed ^ 0x2140).randbytes(SLOT_CAPACITY)
        big = engine.request(big_bytes)
        deadline = time.monotonic() + 30
        while big.state == "out" and time.monotonic() < deadline:
            engine.poll(0.001, max_packets=1)
        in_flight = big.state == "in"
        smalls = [engine.request(b"small-%02d" % n + b"xx") for n in range(small_ops)]
        engine.transmit()
        time.sleep(small_pause_ms / 1e3)
        ahead = None
        while not engine.done([big] + smalls) and time.monotonic() < deadline:
            engine.poll(0.001, max_packets=1)
            if ahead is None and any(s.state == "ok" for s in smalls):
                ahead = big.received() if big.state == "in" else len(big_bytes)
        engine.t.sock.close()
        answer = served(proc, "small-%s" % form)
        before = sum(1 for s in smalls if s.state == "ok" and big.ended is not None and s.ended < big.ended)
        SMALL[form] = {"in_flight": in_flight, "before": before, "big_ok": big.state == "ok" and big.response == big_bytes,
                       "small_ok": sum(s.state == "ok" and s.response.startswith(b"small") for s in smalls),
                       "answer_bytes_before_first_small": ahead}
        print("%-9s the 64 KiB answer in flight when the small requests left: %s, small answers complete before it: %d of %d, %s" % (
            form, in_flight, before, small_ops, SMALL[form]))

    # A record of 16,385 bytes on the sequenced form, and its prefix on the stream.
    TRUNCATED = {}
    for form in ("seqpacket", "stream"):
        path = str(SOCKS / ("trunc-%s.sock" % form))
        proc = serve({"form": form, "path": path, "profile": UNIX_SPEC, "responder": "echo", "connections": 1},
                     "trunc-%s" % form)
        engine = py_client(form, path)
        long_record = bytes(16385)
        if form == "seqpacket":
            engine.t.send_raw(long_record)
        else:
            engine.t.sock.sendall((16385).to_bytes(4, "big") + long_record)
        after = hc.run_requests(engine, [b"after the long record"], window=1, timeout=30)
        engine.t.sock.close()
        conn = served(proc, "trunc-%s" % form)["connections"][0]
        TRUNCATED[form] = {"truncated": conn["truncated"], "link_dropped": conn["link_dropped"],
                           "after_ok": after[0].state == "ok" and after[0].response == b"after the long record"}
        print("%-9s a record of 16,385 bytes: truncated and counted %d, frames dropped %d, the RPC after it: %s" % (
            form, conn["truncated"], conn["link_dropped"], TRUNCATED[form]["after_ok"]))

    # Oversized requests, both directions, both forms.
    ov = random.Random(seed ^ 0x2150)
    OVERSIZED = {}
    WIDE = dict(UNIX_LIMITS, max_message=1_000_000, unsched_bytes=1_000_000, max_rpcs=255)
    for form in ("seqpacket", "stream"):
        sizes = [ov.randint(SLOT_CAPACITY + 1, 80_000) for _ in range(socket_oversized)]
        path = str(SOCKS / ("over-%s.sock" % form))
        proc = serve({"form": form, "path": path, "profile": UNIX_SPEC, "responder": "echo", "connections": 1},
                     "over-%s" % form)
        engine = py_client(form, path, assumed=WIDE)
        rpcs = [engine.request(bytes(n)) for n in sizes]
        # The first packets leave before the server's LIMITS is read, so the
        # server meets the oversized messages and refuses them.
        engine.transmit()
        deadline = time.monotonic() + 60
        while not engine.done(rpcs) and time.monotonic() < deadline:
            engine.poll(0.002)
        engine.t.sock.close()
        conn = served(proc, "over-%s" % form)["connections"][0]
        py_too_long = sum(1 for r in rpcs if r.state in ("failed", "refused") and r.why == "TooLong")
        OVERSIZED[("client.py to serve", form)] = {"too_long": py_too_long, "restarts": engine.stats["restarts"],
                                                   "server_requests": conn["requests"], "server_refused": conn["stats"]["refused"]}

        path = str(SOCKS / ("overpy-%s.sock" % form))
        server = hc.Server(form, path, hc.unix_socket(SLOT_CAPACITY), UNIX_LIMITS)
        box = []
        thread = threading.Thread(target=lambda: box.append(server.serve_one(timeout=60)))
        thread.start()
        answer = unix([{"id": 0, "op": "client", "form": form, "path": path, "profile": UNIX_SPEC, "window": 16,
                        "config": {"page_len": 8192, "pages": 256},
                        "assumed": {"max_message": 1_000_000, "unsched_bytes": 1_000_000, "max_rpcs": 255},
                        "requests": [{"len": n, "fill": {"hex": "00"}} for n in sizes]}], "over-client-%s" % form)[0]
        thread.join(timeout=timeout_s)
        server.close()
        rust_too_long = sum(1 for r in answer["rpcs"] if r["outcome"] in ("failed", "refused") and r["why"] == "TooLong")
        OVERSIZED[("client to client.py", form)] = {"too_long": rust_too_long, "restarts": answer["stats"]["restarts"],
                                                    "server_requests": sum(1 for e in box[0].events if e[0] == "request") if box else None}
    for key, value in OVERSIZED.items():
        print("oversized, %s, %-9s %s" % (key[0], key[1], value))


## The real socket half: the table against AoFrame alone

ADR-0023 asks the plan that owns the carriers to measure Homa against AoFrame alone on each carrier
in bytes on the link, latency, throughput, bytes added to each target and resident memory. At stage
H2 no GraphTP travels, so the Homa column carries messages of the same length as Homa RPCs through
`homa-unix serve` with its engine, and the AoFrame column the same messages whole in prefixed
AoFrame through `homa-unix serve` with `homa` false, first in first out, which on the sequenced form
go in records of at most a packet each. The client is `client.py` for Homa and its `BaselineClient`
for AoFrame alone.

The bytes on the link are those each side wrote for a pause request of 27 bytes up and for a 16 KiB
answer down, the frames of the stream included. The latency is the mean wait of the pause answer
sent while a 16 KiB answer is in flight, from the pause request leaving to its answer, over
`table_draws` draws. The throughput is the goodput of the 16 KiB echo. The bytes added are those
of the device probe of `23_homa_budget`, `homa` and AoFrame against AoFrame alone, and the resident
memory is the storage of the endpoint a connection of `serve` holds against the buffers AoFrame
alone holds for a whole message each way, from `homa-sim sizes`. Every number is the run's own and
printed, since on a local socket both columns take microseconds.


In [ ]:
if half == "socket":
    import re
    import shutil

    PAUSE = b'{"query":"mutation{pause}"}'
    SIXTEEN = random.Random(seed ^ 0x2160).randbytes(16384)
    PROBE = P.tests / "fixtures" / "rs" / "homa-probe" / "Cargo.toml"
    PROBE_DIR = P.target / "homa-probe"
    LLVM_SIZE = shutil.which("llvm-size")

    def probe_bytes(features, target_dir):
        extra = ["--features", features] if features else []
        run([CARGO, "build", "--locked", "--release", "--manifest-path", PROBE, "--target", "riscv32imc-unknown-none-elf",
             "--target-dir", target_dir, *extra], cwd=P.root, timeout_s=900)
        archive = target_dir / "riscv32imc-unknown-none-elf" / "release" / "libhoma_probe.a"
        total, member = 0, None
        for line in run([LLVM_SIZE, "-A", archive], timeout_s=120).out.splitlines():
            head = re.match(r"^(\S+\.o)\s+\(ex ", line)
            if head:
                member = head.group(1).startswith("homa_probe")
                continue
            cells = line.split()
            if member and cells and re.match(r"^\.(text|rodata|srodata)", cells[0]):
                total += int(cells[1])
        return total

    probe_whole = probe_bytes(None, PROBE_DIR)
    probe_frame = probe_bytes("frame-only", PROBE_DIR / "frame-only")
    SERVE_STORAGE = {"peers": 1, "client_slots": 0, "server_slots": 16, "page_len": 8192, "pages": 256, "max_gaps": 4}
    storage = sim([{"id": 0, "op": "sizes", "config": SERVE_STORAGE, "max_packets": [16384, SLOT_CAPACITY]}], "socket-sizes")[0]
    buffers = {b["max_packet"]: b for b in storage["buffers"]}

    def homa_measures(form):
        path = str(SOCKS / ("table-%s.sock" % form))
        proc = serve({"form": form, "path": path, "profile": UNIX_SPEC, "responder": "echo", "reserve": "none",
                      "connections": 1, "send_buffer": serve_send_buffer}, "table-%s" % form)
        engine = py_client(form, path)
        hc.run_requests(engine, [b"warm"], window=1, timeout=30)
        before = dict(engine.t.counts)
        pause = hc.run_requests(engine, [PAUSE], window=1, timeout=30)[0]
        after_pause = dict(engine.t.counts)
        started = time.monotonic()
        big = hc.run_requests(engine, [SIXTEEN], window=1, timeout=30)[0]
        transfer = time.monotonic() - started
        after_big = dict(engine.t.counts)
        waits = []
        for _ in range(table_draws):
            large = engine.request(SIXTEEN)
            while large.state == "out":
                engine.poll(0.0005)
            left = time.monotonic()
            small = engine.request(PAUSE)
            while small.state in ("out", "in"):
                engine.poll(0.0005)
            waits.append(time.monotonic() - left)
            while large.state in ("out", "in"):
                engine.poll(0.0005)
        engine.t.sock.close()
        served(proc, "table-%s" % form)
        assert pause.state == "ok" and big.state == "ok" and big.response == SIXTEEN
        return {
            "bytes up, a pause": after_pause["link_bytes_out"] - before["link_bytes_out"],
            "bytes down, the pause answer": after_pause["link_bytes_in"] - before["link_bytes_in"],
            "bytes down, 16 KiB": after_big["link_bytes_in"] - after_pause["link_bytes_in"],
            "wait behind 16 KiB, ms": 1e3 * sum(waits) / len(waits),
            "goodput, bytes a second": 16384 / transfer,
            "bytes added to the target": probe_whole,
            "resident memory, bytes": storage["storage_bytes"]["unit"],
        }

    def aoframe_measures(form):
        path = str(SOCKS / ("base-%s.sock" % form))
        proc = serve({"form": form, "path": path, "profile": UNIX_SPEC, "responder": "echo", "homa": False,
                      "connections": 1, "send_buffer": serve_send_buffer}, "base-%s" % form)
        base = hc.BaselineClient(form, path, 16384, SLOT_CAPACITY)
        base.send(b"warm")
        base.receive()
        before = dict(base.counts)
        base.send(PAUSE)
        assert base.receive() == PAUSE
        after_pause = dict(base.counts)
        started = time.monotonic()
        base.send(SIXTEEN)
        assert base.receive() == SIXTEEN
        transfer = time.monotonic() - started
        after_big = dict(base.counts)
        waits = []
        for _ in range(table_draws):
            base.send(SIXTEEN)
            left = time.monotonic()
            base.send(PAUSE)
            first = base.receive()
            second = base.receive()
            waits.append(time.monotonic() - left)
            assert first == SIXTEEN and second == PAUSE, "AoFrame alone answers first in first out"
        base.close()
        served(proc, "base-%s" % form)
        return {
            "bytes up, a pause": after_pause["link_bytes_out"] - before["link_bytes_out"],
            "bytes down, the pause answer": after_pause["link_bytes_in"] - before["link_bytes_in"],
            "bytes down, 16 KiB": after_big["link_bytes_in"] - after_pause["link_bytes_in"],
            "wait behind 16 KiB, ms": 1e3 * sum(waits) / len(waits),
            "goodput, bytes a second": 16384 / transfer,
            "bytes added to the target": probe_frame,
            "resident memory, bytes": buffers[SLOT_CAPACITY]["prefixed"] + buffers[SLOT_CAPACITY]["frame_prefixed"],
        }

    TABLE = {}
    for form in ("seqpacket", "stream"):
        TABLE[form] = {"Homa": homa_measures(form), "AoFrame alone": aoframe_measures(form)}
        print(form)
        print("  %-30s %-16s %-16s" % ("row", "Homa", "AoFrame alone"))
        for label in TABLE[form]["Homa"]:
            homa, alone = TABLE[form]["Homa"][label], TABLE[form]["AoFrame alone"][label]
            fmt = lambda v: ("%.3f" % v) if isinstance(v, float) else str(v)
            print("  %-30s %-16s %-16s" % (label, fmt(homa), fmt(alone)))
    TABLE_COMPLETE = sum(1 for form in TABLE if all(v is not None for column in TABLE[form].values() for v in column.values())
                         and len(TABLE[form]["Homa"]) == 7)
    print("tables complete: %d of 2" % TABLE_COMPLETE)


## The halves this entry does not run

Each half runs as its own entry of the matrix, and the halves this entry does not name print not
run.


In [ ]:
HALVES = {
    "compact": "its own entry of the matrix, 21_homa_loopback-compact",
    "network": "its own entry of the matrix, 21_homa_loopback-network",
    "socket": "its own entry of the matrix, 21_homa_loopback-socket",
}
NAMES = {"compact": "compact half", "network": "Network half", "socket": "real socket half"}
for other, why in HALVES.items():
    if other != half:
        print("not run: %s, which runs as %s" % (NAMES[other], why))

## Verdict

In [ ]:
print("panics or crashes of the seam:", len(PANICS))
for panic in PANICS:
    print("  ", panic)
assert not PANICS, PANICS
if half == "compact":
    capacity, binding = RUNS["capacity"], RUNS["binding"]
    for bound, got in RUNS.items():
        results = got["results"]
        print("%s bound: RPCs %d, payload differences %d, not answered %d, delivered twice %d,"
              " reserved %d, replied whole %d" % (
                  bound, total, results["payload_differences"], results["not_ok"], results["duplicates"],
                  results["reserved"], results["replied"]))
        print("%s bound: transmit decisions %d, against SRPT outside the FIFO share %d,"
              " granted and not received above window %d" % (
                  bound, got["srpt"]["decisions"], got["srpt"]["against"], got["grants"]["window_over"]))
        print("%s bound: GRANTs %d, to the message SRPT names %d, to the oldest where SRPT names"
              " another %d against the %d the FIFO share allows, receiving sides above their share %d,"
              " to any other message %d, of a message the trace holds no DATA of %d" % (
                  bound, got["grant_order"]["grants"], got["grant_order"]["by_srpt"],
                  got["grant_order"]["by_fifo"], got["grant_order"]["fifo_share"],
                  got["grant_order"]["share_exceeded"], got["grant_order"]["against"],
                  got["grant_order"]["not_judged"]))
    print("expected above the max_incoming of the design at any row, unscheduled bytes included:",
          capacity["grants"]["expected_over_any_row"])
    print("expected above the binding max_incoming at a GRANT:", binding["grants"]["incoming_over"],
          " at any row, unscheduled bytes included:", binding["grants"]["expected_over_any_row"])
    assert not PANICS, PANICS
    for bound, got in RUNS.items():
        results = got["results"]
        assert results["payload_differences"] == 0 and results["not_ok"] == 0 and results["duplicates"] == 0, (bound, results)
        assert got["srpt"]["against"] == 0 and got["srpt"]["share_exceeded"] == 0, (bound, got["srpt"])
        assert got["grants"]["window_over"] == 0, (bound, got["grants"])
        assert got["grant_order"]["against"] == 0 and got["grant_order"]["share_exceeded"] == 0, (bound, got["grant_order"])
    assert fifo_only["against"] == 0 and fifo_only["share_exceeded"] == ["B"], fifo_only
    assert capacity["grants"]["expected_over_any_row"] == 0, capacity["grants"]
    assert binding["grants"]["incoming_over"] == 0, binding["grants"]
    assert oversized_failed == oversized_ops and oversized_restarts == 0, (too_long, other)
    assert reserve_differences == 0
    assert returned == reservation_ops and dropped_ok == reservation_ops
    print()
    print("VERDICT: %d RPCs under each of two bounds with 0 payload differences, 0 of %d and %d"
          " transmit decisions against SRPT outside the FIFO share, every GRANT to the message SRPT"
          " or the FIFO share names with the GRANTs to the oldest within the share, granted bytes"
          " within window,"
          " expected bytes within the max_incoming of the design at every row and within the"
          " binding max_incoming at every GRANT, %d of %d oversized requests failed TooLong with %d"
          " restarts, 0 byte differences between reserved and whole answers, %d of %d dropped"
          " reservations returned their pages, 0 panics, and the Network and real socket halves"
          " not run" % (
              total, capacity["srpt"]["decisions"], binding["srpt"]["decisions"], oversized_failed,
              oversized_ops, oversized_restarts, returned, reservation_ops))
if half == "network":
    for (bound, mode), got in NET_RUNS.items():
        print("%s bound, %s: RPCs %d, payload differences %d, not answered %d, delivered twice %d" % (
            bound, mode, got["rpcs"], got["payload_differences"], got["not_ok"], got["duplicates"]))
        print("%s bound, %s: transmit decisions %d, against SRPT %d, GRANTs %d, against the scheduler %d,"
              " decisions %d of which the FIFO share %d, beyond the window %d, off the dynamic window %d,"
              " off homa_grant_priority %d, expected above max_incoming at a decision %d, at any row %d" % (
                  bound, mode, got["decisions_tx"], got["against"], got["grants"], got["grant_against"],
                  got["decisions"], got["fifo"], got["window_over"], got["window_formula"],
                  got["level_mismatch"], got["expected_over"], got["expected_over_any_row"]))
    for (bound, mode), got in NET_RUNS.items():
        assert got["payload_differences"] == 0 and got["not_ok"] == 0 and got["duplicates"] == 0, (bound, mode, got)
        assert got["against"] == 0 and got["share_exceeded"] == 0, (bound, mode, got)
        assert got["grant_against"] == 0 and got["grant_share_exceeded"] == 0, (bound, mode, got)
        assert got["window_over"] == 0 and got["window_formula"] == 0, (bound, mode, got)
        assert got["level_mismatch"] == 0 and got["expected_over"] == 0, (bound, mode, got)
        if bound == "profile":
            assert got["expected_over_any_row"] == 0, (bound, mode, got)
            assert got["rpcs"] == max_ops, got
    assert NET_RUNS[("profile", "all-or-nothing")]["start_msg"] > 0
    assert net_oversized_failed == oversized_ops and net_oversized_restarts == 0, (net_too_long, net_other)
    assert net_reserve_differences == 0
    assert net_returned == reservation_ops and net_dropped_ok == reservation_ops
    assert len(seen) == client_ports and per_peer == [rpcs_per_port] * client_ports and apart, seen
    assert ports_seen == list(range(4000, 4000 + client_ports)), ports_seen
    assert all(v == (rpcs_per_port, rpcs_per_port) for v in answered_by_port.values()), answered_by_port
    assert ports_answer["foreign_counted"] == foreign_packets == ports_answer["server_stats"]["wrong_port"]
    assert outside_port == outside_requests
    assert ports_answer["down_case"] == {"first_granted_first": True, "link_down": True, "second_whole": True}
    assert table["inserted"] == table_peers and table["admitted"] == table_peers, table
    print()
    print("VERDICT: %d RPCs in each scheduling mode on the Network profile with 0 payload differences,"
          " 0 transmit decisions against SRPT, every GRANT to the message the scheduler names, every"
          " decision within its dynamic window and at the level of homa_grant_priority, expected"
          " bytes within max_incoming at every row and within the binding bound at every decision,"
          " %d of %d oversized requests failed TooLong with %d restarts, 0 byte differences between"
          " reserved and whole answers, %d of %d dropped reservations returned their pages, the"
          " longest probe of the table at %d peers %d, %d source ports seen as %d peers with their"
          " RPCs apart, %d of %d packets to a foreign port counted in wrong_port, %d of %d requests"
          " from outside the client's ports failed Port, 0 panics, and the compact and real socket"
          " halves not run" % (
              max_ops, net_oversized_failed, oversized_ops, net_oversized_restarts, net_returned,
              reservation_ops, table_peers, table["longest_probe"], client_ports, len(seen),
              ports_answer["foreign_counted"], foreign_packets, outside_port, outside_requests))
if half == "socket":
    for key, got in SOCKET_RESULTS.items():
        print("%s, %s: %s" % (key[0], key[1], got))
    one = [SOCKET_RESULTS[("one process", link)] for link in ("seqpacket", "stream")]
    two = [SOCKET_RESULTS[(side, form)] for side in ("serve against client.py", "client against client.py")
           for form in ("seqpacket", "stream")]
    for got in one:
        assert got["rpcs"] == socket_ops and got["payload_differences"] == 0 and got["not_ok"] == 0 and got["duplicates"] == 0, got
    for got in two:
        assert got["rpcs"] == socket_ops and got["ok"] == got["rpcs"] and got["payload_differences"] == 0, got
        assert got.get("server_duplicates", 0) in (0, None) and got["restarts"] == 0, got
    assert all(SMALL[form]["in_flight"] and SMALL[form]["before"] == small_ops and SMALL[form]["big_ok"]
               and SMALL[form]["small_ok"] == small_ops for form in SMALL), SMALL
    assert TRUNCATED["seqpacket"]["truncated"] == 1 and TRUNCATED["seqpacket"]["after_ok"], TRUNCATED
    assert TRUNCATED["stream"]["link_dropped"] == 1 and TRUNCATED["stream"]["after_ok"], TRUNCATED
    assert all(v["too_long"] == socket_oversized and v["restarts"] == 0 for v in OVERSIZED.values()), OVERSIZED
    assert TABLE_COMPLETE == 2
    print()
    print("VERDICT: %d RPCs over homa::unix::pair and %d over a StreamListener connection in one process,"
          " %d RPCs on each form in each direction across two processes, homa-unix serve against"
          " client.py and homa-unix client against a server of client.py, 0 payload differences, 0"
          " duplicate deliveries, %d of %d small answers before the 64 KiB answer on each form, 1"
          " truncated record of 16,385 bytes dropped and counted, %d of %d oversized requests failed"
          " TooLong in each direction and form with 0 restarts, 0 panics, the table against AoFrame"
          " alone complete for both forms, and the compact and Network halves not run" % (
              socket_ops, socket_ops, socket_ops, small_ops, small_ops, socket_oversized, socket_oversized))